# Film collection: familiar films from five decades

Read the film list, fetch HTML and API data, check the match, then save.

The list uses well-known films from ranking references. It is not a random sample; conclusions will apply to the chosen films.

Start Jupyter in the project root. Run top to bottom. Saved responses are used by default. After editing `film_list.json`, set `live = True` to collect a new batch. Check the sites' robots.txt and terms before live collection.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re
import time
import hashlib

folder = Path.cwd()
live = False
films = json.loads((folder / "film_list.json").read_text(encoding="utf-8"))["films"]
base = folder / "01_raw/ranked_films"
latest = base / "latest_run.txt"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") if live else latest.read_text().strip()
run = base / run_id
run.mkdir(parents=True, exist_ok=True)
headers = {"User-Agent": "ADS206-CW1-film-collection/0.3 (educational, limited requests)"}
if live:
    (run / "film_list.json").write_bytes((folder / "film_list.json").read_bytes())
print("Mode:", "live requests" if live else "saved responses")
pd.DataFrame(films)[["title", "title_zh", "year", "rank_source", "reference_rank"]]


Mode: saved responses


## 1. Request and save

Keep raw responses and a request log. Use a timeout, a pause and up to three attempts.

In [2]:
def fetch(url, name, params=None):
    if not live:
        return (run / name).read_bytes()
    for attempt in range(3):
        started = datetime.now(timezone.utc).isoformat()
        clock = time.perf_counter()
        reply = None
        log = {"time_utc": started, "url": url, "params": params,
               "retry_count": attempt, "collector": "AI-assisted film collection"}
        try:
            reply = requests.get(url, params=params, headers=headers, timeout=30)
            tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
            file = f"{Path(name).stem}_{tag}{Path(name).suffix}"
            (run / file).write_bytes(reply.content)
            log.update(url=reply.url, status=reply.status_code, file=file,
                       retry_after=reply.headers.get("Retry-After"),
                       redirects=[r.status_code for r in reply.history],
                       sha256=hashlib.sha256(reply.content).hexdigest())
        except requests.RequestException as error:
            log["error"] = type(error).__name__
        log["seconds"] = round(time.perf_counter() - clock, 3)
        with (run / "requests.jsonl").open("a", encoding="utf-8") as file:
            file.write(json.dumps(log) + "\n")
        time.sleep(5)
        if reply is not None and reply.ok:
            (run / name).write_bytes(reply.content)
            return reply.content
        if reply is not None and reply.status_code != 429 and reply.status_code < 500:
            reply.raise_for_status()
        if attempt == 2:
            raise ConnectionError("Request failed. See requests.jsonl.")
        delay = 2 ** attempt
        if reply is not None and reply.status_code == 429:
            value = reply.headers.get("Retry-After", "")
            if not value.isdigit() or int(value) > 60:
                raise RuntimeError("Rate limited. Stop and check Retry-After.")
            delay = max(delay, int(value))
        time.sleep(delay)


## 2. Read the HTML pages

Use the links in the list. Read the heading, IMDb links, overall shot fields and contributor comment. Keep unavailable titles on the candidate list.

In [3]:
html_rows = {}
for film in films:
    if not film["html_url"]:
        print("Page not found yet:", film["title"])
        continue
    print("Reading HTML:", film["title"], flush=True)
    page_id = film["html_url"].rstrip("/").split("/")[-1]
    soup = BeautifulSoup(fetch(film["html_url"], page_id + ".html"), "html.parser")
    heading = soup.find("header").get_text(" ", strip=True)
    title = re.search(r"^(.*?)\s*\(\s*(\d{4})", heading)
    assert title, "Check the heading: " + film["title"]
    stats = {}
    for button in soup.select("button"):
        parts = button.find_all("span", recursive=False)
        if len(parts) == 2:
            label = parts[0].get_text(strip=True).rstrip(":")
            if label in {"NoS", "LEN", "ASL", "MSL"}:
                assert label not in stats, "Repeated overall field."
                stats[label] = parts[1].get_text(strip=True)
    assert len(stats) == 4, "Check the shot fields: " + film["title"]
    ids = {re.search(r"tt\d+", a["href"]).group()
           for a in soup.find_all("a", href=re.compile(r"imdb\.com/title/tt\d+"))}
    comment = re.search(r"comment:\s*(.*?)\s+NoS\s*:", soup.get_text(" ", strip=True), re.I)
    html_rows[film["wikidata_id"]] = {"title_html": title[1].strip(),
        "year_html": int(title[2]), "imdb_ids_html": sorted(ids),
        "shot_count": int(stats["NoS"].replace(",", "")), "measured_duration": stats["LEN"],
        "asl_seconds": float(stats["ASL"]), "msl_seconds": float(stats["MSL"]),
        "comment": comment[1] if comment else "", "html_url": film["html_url"]}
print("HTML records read:", len(html_rows))


Reading HTML: Singin' in the Rain


Reading HTML: Roman Holiday


Reading HTML: Seven Samurai


Reading HTML:

 12 Angry Men


Page not found yet: Witness for the Prosecution
Reading HTML: Psycho


... (22 outputs left out)

Reading HTML: Life Is Beautiful


Reading HTML: Titanic


HTML records read:

 22


## 3. Read API metadata

P345: IMDb IDs; P577: release dates; P57: directors; P495: countries; P136: genres. Keep multiple values. P2047 runtimes below use minutes (Q7727).

In [4]:
def values(item, prop):
    rows = item.get("statements", {}).get(prop, [])
    return [row["value"]["content"] for row in rows if row.get("rank") != "deprecated"
            and row.get("value", {}).get("type") == "value"]

api_rows = {}
for film in films:
    uid = film["wikidata_id"]
    if uid not in html_rows:
        continue
    print("Reading API:", film["title"], flush=True)
    url = "https://www.wikidata.org/w/rest.php/wikibase/v1/entities/items/" + uid
    item = json.loads(fetch(url, uid + ".json"))
    assert item["id"] == uid
    years = sorted({int(date["time"][1:5]) for date in values(item, "P577")
        if date.get("precision", 0) >= 9 and re.match(r"^\+\d{4}-", date["time"])})
    api_rows[uid] = {"wikidata_id": uid,
        "title_api": item.get("labels", {}).get("en", item.get("labels", {}).get("mul", "")),
        "description_api": item.get("descriptions", {}).get("en", ""),
        "imdb_ids_api": values(item, "P345"), "release_years_api": years,
        "director_ids": values(item, "P57"), "country_ids": values(item, "P495"),
        "genre_ids": values(item, "P136"), "api_url": url,
        "runtime_minutes_api": sorted({float(v["amount"]) for v in values(item, "P2047")
            if v.get("unit", "").endswith("/Q7727")})}
print("API records read:", len(api_rows))


Reading API: Singin' in the Rain


Reading API: Roman Holiday


Reading API: Seven Samurai


Reading API: 12 Angry Men


Reading API: Psycho


Reading API: The Apartment


... (15 outputs left out)

 The Shawshank Redemption


Reading API: Life Is Beautiful


Reading API: Titanic


API records read: 22


## 4. Read the metadata names

Look up English labels for country, director and genre IDs, at most 50 IDs per request.

In [5]:
ids = sorted({uid for row in api_rows.values() for field in ["director_ids", "country_ids", "genre_ids"]
              for uid in row[field]})
names = {}
for start in range(0, len(ids), 50):
    params = {"action": "wbgetentities", "ids": "|".join(ids[start:start + 50]),
              "props": "labels", "languages": "en|mul", "format": "json"}
    data = json.loads(fetch("https://www.wikidata.org/w/api.php", f"labels_{start}.json", params))
    for uid, item in data["entities"].items():
        labels = item.get("labels", {})
        names[uid] = labels.get("en", labels.get("mul", {"value": uid}))["value"]
for row in api_rows.values():
    row["directors"] = [names.get(uid, uid) for uid in row["director_ids"]]
    row["countries"] = [names.get(uid, uid) for uid in row["country_ids"]]
    row["genres"] = [names.get(uid, uid) for uid in row["genre_ids"]]


## 5. Match, combine and save

Check IMDb IDs and year. Keep uncertain matches marked for review. Identity and full-film boundaries are separate checks. These are candidate records; the final sample and article text are still pending.

In [6]:
records = []
for film in films:
    uid = film["wikidata_id"]
    if uid not in html_rows or uid not in api_rows:
        continue
    page, item = html_rows[uid], api_rows[uid]
    shared = bool(set(page["imdb_ids_html"]) & set(item["imdb_ids_api"]))
    year_ok = page["year_html"] == film["year"] and film["year"] in item["release_years_api"]
    record = {**film, **page, **item, "identity_check": "IMDb + year" if shared and year_ok else "review",
        "measurement_check": "review boundaries", "selected_for_main_analysis": False,
        "evidence_run": run_id}
    if re.search(r"first 30|min only|\bscene\b|\bextract\b|\bclip\b|half", page["comment"], re.I):
        record["measurement_check"] = "possible excerpt"
    records.append(record)
result = {"stage": "rank-referenced candidates; final sample pending", "run_id": run_id,
          "collector": "AI-assisted acquisition; student review required", "records": records}
if live:
    body = json.dumps(result, ensure_ascii=False, indent=2)
    (run / "records.json").write_text(body, encoding="utf-8")
    (folder / "02_audit/ranked_films.json").write_text(body, encoding="utf-8")
    latest.write_text(run_id, encoding="utf-8")
print("Candidate pages read:", len(records))
pd.DataFrame(records)[["title", "title_zh", "year", "identity_check", "measurement_check"]]


Candidate pages read: 22


,title,title_zh,year,identity_check,measurement_check
0,Singin' in the Rain,雨中曲,1952,review,review boundaries
1,Roman Holiday,罗马假日,1953,review,review boundaries
2,Seven Samurai,七武士,1954,IMDb + year,review boundaries
3,12 Angry Men,十二怒汉,1957,review,review boundaries
4,Psycho,惊魂记,1960,IMDb + year,possible excerpt
5,The Apartment,桃色公寓,1960,IMDb + year,review boundaries
6,The Sound of Music,音乐之声,1965,review,review boundaries
7,2001: A Space Odyssey,2001太空漫游,1968,IMDb + year,review boundaries
8,The Godfather,教父,1972,review,possible excerpt
9,The Godfather Part II,教父2,1974,IMDb + year,review boundaries


### Saved quality-check example

The earlier Top Gun record measured **01:16.9** and its comment said **Volleyball scene**. The linked film runtime was **109 minutes**. This shows why identity and measurement scope are separate checks. See the saved [earlier review](02_audit/film_review.json).

Sources: [Cinemetrics](https://cinemetrics.uchicago.edu/), [Wikidata REST API](https://www.wikidata.org/wiki/Wikidata:REST_API), and the ranking links in `film_list.json`.